# Preprocessing Dataset Kotor: ICU Survival Prediction

**Mata kuliah:** Data Wrangling | **Dataset:** Kaggle - ML League 2026 Task 2: ICU Survival Prediction
**Target:** `survived_icu` (1 = selamat, 0 = tidak selamat) | **Model supervised:** Logistic Regression

### Tujuan
1. Menemukan semua jenis "kekotoran" pada dataset (bukan hanya missing value).
2. Membersihkan dan mentransformasi data dengan teknik-teknik dari materi kuliah.
3. Membuktikan dengan model supervised (accuracy, dll.) bagaimana pengaruh tiap keputusan preprocessing.

### Kenapa hanya satu model (Logistic Regression)?
- Sederhana dan mudah dijelaskan: tiap fitur punya satu koefisien yang bisa dibaca.
- Peka terhadap kualitas data (skala, outlier, missing), jadi efek preprocessing lebih mudah terlihat.
- Fokus tugas ini adalah **preprocessing**, bukan membandingkan banyak model. Satu model yang sama dipakai di semua eksperimen supaya perbandingannya adil.

### Alur kerja

```
Data mentah -> Audit kualitas data -> Cleaning tahap 1 (aturan pasti, sebelum split)
   -> Split train/test -> Cleaning tahap 2 (diuji pada data latih)
   -> Transformasi -> Reduksi data -> Pipeline akhir -> Evaluasi
```

### Peta materi slide

| Materi di slide | Bagian |
|---|---|
| What is Data, tipe atribut (nominal/ordinal/interval/rasio), diskrit vs kontinu | 2 |
| Data understanding (relevansi), data quality (accuracy, completeness, consistency) | 2, 3 |
| Cleaning: noise, duplicate, missing value, outlier | 3, 4, 6 |
| Transformation: encoding, scaling, smoothing, aggregation, discretization | 7 |
| Feature engineering (extraction, encoding, scaling, selection, combination) | 7.6, 8 |
| Data reduction: variable selection dan variable reduction | 8 |
| Modeling dan evaluasi | 5, 9 |

> **Catatan untuk presentasi:** sel bertanda **(utilitas)** hanya berisi definisi fungsi dan boleh dilewati saat presentasi.

## 1. Import Library dan Data Gathering

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer
from sklearn.preprocessing import (StandardScaler, MinMaxScaler, RobustScaler,
                                   PowerTransformer, QuantileTransformer,
                                   FunctionTransformer, OneHotEncoder,
                                   LabelEncoder, OrdinalEncoder, KBinsDiscretizer)
from sklearn.feature_selection import VarianceThreshold, SelectKBest, mutual_info_classif
from sklearn.decomposition import PCA
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)
RANDOM_STATE = 42

In [ ]:
# Data Gathering. Sesuaikan nama file jika perlu; kode ini mencoba beberapa nama umum.
CANDIDATES = ["data_icu.csv", "train (6).csv", "train.csv"]
DATA_PATH = next(p for p in CANDIDATES if os.path.exists(p))

# na_values: berbagai penulisan "kosong" (n/a, NA, --, dst. seperti di slide) dibaca sebagai NaN
NA_VALUES = ["", " ", "NA", "N/A", "n/a", "na", "NaN", "nan", "NULL", "null", "None", "-", "--", "?"]

df = pd.read_csv(DATA_PATH, na_values=NA_VALUES)
print("File:", DATA_PATH, "| ukuran:", df.shape)
df.head()

## 2. Data Understanding

Pertanyaan relevansi dari slide, dijawab untuk dataset ini:

| Pertanyaan | Jawaban |
|---|---|
| Data apa yang tersedia? | Data pasien ICU: demografi, tanda vital, hasil lab, skor keparahan, riwayat penyakit |
| Berapa banyak? | 4.580 pasien, 16 kolom |
| Ada label (target)? | Ada: `survived_icu` |
| Relevan? | Ya, memprediksi kelangsungan hidup pasien ICU |
| Bagaimana kualitasnya? | Kotor, lihat bagian 3 |
| Ada data eksternal? | Tidak. Rentang fisiologis normal memakai pengetahuan medis umum |

In [ ]:
df.info()

In [ ]:
df.describe().T

**Cara membaca:** bila rata-rata (mean) jauh dari median (50%), curigai nilai ekstrem. Contoh: `heart_rate` rata-rata 99 tetapi median 89 dengan maksimum 1207, dan `systolic_bp` rata-rata 133 tetapi median 118.

### 2.1 Jenis Atribut (nominal, ordinal, interval, rasio; diskrit vs kontinu)

In [ ]:
attr_types = pd.DataFrame([
    ("patient_id",           "Nominal",  "Diskrit",  "Identitas pasien, tidak bermakna numerik -> dibuang dari fitur"),
    ("age",                  "Rasio",    "Diskrit*", "Tahun; nol absolut ada (*tercatat bilangan bulat)"),
    ("gender",               "Nominal",  "Diskrit",  "Biner (0/1), angka hanya simbol"),
    ("bmi",                  "Rasio",    "Kontinu",  "kg/m2"),
    ("heart_rate",           "Rasio",    "Kontinu",  "denyut/menit"),
    ("systolic_bp",          "Rasio",    "Kontinu",  "mmHg"),
    ("diastolic_bp",         "Rasio",    "Kontinu",  "mmHg"),
    ("respiratory_rate",     "Rasio",    "Kontinu",  "napas/menit"),
    ("oxygen_saturation",    "Rasio",    "Kontinu",  "persen, dibatasi maksimum 100"),
    ("glucose",              "Rasio",    "Kontinu",  "mg/dL"),
    ("creatinine",           "Rasio",    "Kontinu",  "mg/dL"),
    ("wbc_count",            "Rasio",    "Kontinu",  "ribu sel/uL"),
    ("severity_score",       "Ordinal",  "Diskrit",  "Skor 0-24; selisih antar skor belum tentu setara"),
    ("ventilation_required", "Nominal",  "Diskrit",  "Biner (0/1)"),
    ("chronic_conditions",   "Rasio",    "Diskrit",  "Jumlah penyakit kronis (0-5)"),
    ("survived_icu",         "Nominal",  "Diskrit",  "TARGET, biner"),
], columns=["Kolom", "Tipe skala", "Diskrit/Kontinu", "Catatan"])
attr_types

Tidak ada atribut bertipe **interval** (seperti suhu Celsius, tahun kalender, atau IQ) pada dataset ini.

> **Poin presentasi:** 16 kolom; hampir semuanya numerik rasio. Hanya `severity_score` yang ordinal, dan `gender`, `ventilation_required`, serta target yang nominal biner.

## 3. Audit Kualitas Data

Tiga elemen kualitas data menurut slide dan temuan yang terkait:

| Elemen | Arti | Temuan pada dataset ini |
|---|---|---|
| **Accuracy** | Nilai sesuai kenyataan | Nilai mustahil secara medis (noise), outlier ekstrem |
| **Completeness** | Data lengkap | Missing value, berkelompok pada 5 kolom |
| **Consistency** | Konsisten antar atribut | Tekanan diastolik >= sistolik, duplikat tersembunyi |

In [ ]:
MISS_COLS = ["bmi", "respiratory_rate", "oxygen_saturation", "creatinine", "wbc_count"]
MEASURED  = ["bmi", "heart_rate", "systolic_bp", "diastolic_bp", "respiratory_rate",
             "oxygen_saturation", "glucose", "creatinine", "wbc_count"]
NUM_COLS  = ["age"] + MEASURED + ["severity_score"]
FEATURES  = [c for c in df.columns if c not in ["patient_id", "survived_icu"]]

### 3.1 Missing Value

In [ ]:
missing = pd.DataFrame({"jumlah_missing": df.isna().sum(),
                        "persen": (df.isna().mean() * 100).round(2)}).sort_values("jumlah_missing", ascending=False)
print(missing[missing["jumlah_missing"] > 0])
print("\nPasien dengan minimal satu missing:", df.isna().any(axis=1).sum())

In [ ]:
# Pola missing: kolom mana saja yang kosong bersamaan?
pola = df[MISS_COLS].isna().apply(
    lambda r: ", ".join([c for c in MISS_COLS if r[c]]) or "Tidak ada missing", axis=1)
print(pola.value_counts().rename("jumlah").to_frame())

plt.figure(figsize=(8, 3.5))
plt.imshow(df[MISS_COLS].isna().values[:300].T, aspect="auto", cmap="gray_r", interpolation="nearest")
plt.yticks(range(len(MISS_COLS)), MISS_COLS)
plt.xlabel("Baris (300 pertama)")
plt.title("Peta missing value (hitam = kosong)")
plt.show()

**Temuan:** semua baris yang punya missing pasti `wbc_count`-nya kosong, dan ada satu blok besar di mana lima kolom kosong **sekaligus** (data tidak tercatat sebagai satu paket).

#### Jenis missing: MCAR, MAR, atau MNAR?
- **MCAR**: kosong acak, tidak berhubungan dengan variabel apa pun
- **MAR**: kosongnya berhubungan dengan variabel lain yang teramati
- **MNAR**: kosongnya berhubungan dengan nilai yang hilang itu sendiri

Cara menguji: bandingkan baris yang punya missing dengan baris yang lengkap pada variabel lain dan pada target.

In [ ]:
tmp = df.copy()
tmp["ada_missing"] = tmp[MISS_COLS].isna().any(axis=1)

rows = []
for c in ["age", "severity_score", "heart_rate", "glucose", "ventilation_required",
          "chronic_conditions", "gender", "survived_icu"]:
    a = tmp.loc[tmp["ada_missing"], c].dropna()
    b = tmp.loc[~tmp["ada_missing"], c].dropna()
    t, p = stats.ttest_ind(a, b, equal_var=False)
    rows.append({"variabel": c, "mean_baris_missing": round(a.mean(), 3),
                 "mean_baris_lengkap": round(b.mean(), 3), "p_value": round(p, 4)})
pd.DataFrame(rows).set_index("variabel")

**Interpretasi:** tidak ada p-value di bawah 0,05, jadi tidak ada bukti bahwa missing bergantung pada variabel lain (bukan MAR) atau pada target (tidak ada petunjuk MNAR). Pola ini **konsisten dengan MCAR**. Catatan: MCAR tidak bisa dibuktikan 100% dari data; uji ini hanya mendukung atau membantahnya.

> **Poin presentasi:** 5 kolom punya missing (13-18%), kosongnya berkelompok, dan tampak acak (MCAR).

### 3.2 Duplikat

In [ ]:
print("Duplikat seluruh kolom (termasuk ID)  :", df.duplicated().sum())
print("Duplikat fitur+target (tanpa patient_id):", df.duplicated(subset=FEATURES + ["survived_icu"]).sum(), "baris tambahan")
print("Baris yang terlibat duplikat           :", df.duplicated(subset=FEATURES + ["survived_icu"], keep=False).sum())
print("patient_id yang sama                   :", df["patient_id"].duplicated().sum())

# Pola penulisan ID (regex)
id_normal = df["patient_id"].str.match(r"^ICU_\d{5}$")
id_dup = df["patient_id"].str.match(r"^ICU_DUP_\d{4}$")
print("\nID berpola ICU_xxxxx    :", id_normal.sum())
print("ID berpola ICU_DUP_xxxx :", id_dup.sum())
print("ID tidak dikenal        :", (~(id_normal | id_dup)).sum())

**Temuan:**
- Duplikat "biasa" tidak terdeteksi (0) karena `patient_id` setiap baris berbeda. Baru terlihat setelah ID diabaikan: 58 baris tambahan.
- 80 ID berpola `ICU_DUP_xxxx` adalah petunjuk bahwa ada baris sisipan. Tetapi hanya 58 yang sama persis dengan baris lain. Sisanya (22) perlu diperiksa di bagian 4.1.

### 3.3 Noise dan Outlier

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 9))
for ax, c in zip(axes.ravel(), NUM_COLS):
    ax.boxplot(df[c].dropna())
    ax.set_title(c)
for ax in axes.ravel()[len(NUM_COLS):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Rentang fisiologis yang masuk akal (pengetahuan medis umum)
RANGES = {
    "age": (18, 100), "bmi": (12, 60), "heart_rate": (30, 220),
    "systolic_bp": (60, 250), "diastolic_bp": (30, 150), "respiratory_rate": (5, 60),
    "oxygen_saturation": (50, 100), "glucose": (30, 600), "creatinine": (0.1, 15),
    "wbc_count": (0.5, 60),
}
rows = []
for c, (lo, hi) in RANGES.items():
    q1, q3 = df[c].quantile([0.25, 0.75]); iqr = q3 - q1
    rows.append({"kolom": c, "min": df[c].min(), "max": df[c].max(), "rentang_valid": f"{lo}-{hi}",
                 "di_luar_rentang": int(((df[c] < lo) | (df[c] > hi)).sum()),
                 "outlier_IQR": int(((df[c] < q1 - 1.5 * iqr) | (df[c] > q3 + 1.5 * iqr)).sum()),
                 "skewness": round(df[c].skew(), 2)})
pd.DataFrame(rows).set_index("kolom")

**Temuan:** `heart_rate`, `systolic_bp`, `glucose`, dan `creatinine` punya nilai ekstrem yang mustahil (denyut > 1000/menit, glukosa > 2000). Pertanyaannya: apakah ini outlier acak, atau satu kelompok baris yang rusak bersamaan? Cek dengan melihat apakah ekstremnya muncul di baris yang sama.

In [ ]:
extreme_hr = df["heart_rate"] > 220
extreme_sbp = df["systolic_bp"] > 250
print("HR > 220      :", extreme_hr.sum(), "baris")
print("SBP > 250     :", extreme_sbp.sum(), "baris")
print("Keduanya      :", (extreme_hr & extreme_sbp).sum(), "baris  <- baris yang sama")

# Apakah ada 'jurang' antara data normal dan data ekstrem?
print("\nHR  tertinggi di kelompok normal :", df.loc[~extreme_hr, "heart_rate"].max())
print("HR  terendah di kelompok ekstrem :", round(df.loc[extreme_hr, "heart_rate"].min(), 1))
print("SBP tertinggi di kelompok normal :", df.loc[~extreme_sbp, "systolic_bp"].max())
print("SBP terendah di kelompok ekstrem :", round(df.loc[extreme_sbp, "systolic_bp"].min(), 1))

In [ ]:
# Sensitivitas ambang: apakah hasilnya bergantung pada angka batas yang dipilih?
thr = [180, 200, 220, 250, 300]
pd.DataFrame({"ambang": thr,
              "baris HR > ambang": [(df["heart_rate"] > t).sum() for t in thr],
              "baris SBP > ambang": [(df["systolic_bp"] > t).sum() for t in thr]}).set_index("ambang")

**Temuan penting:**
- Jumlah baris ekstrem **sama (92) untuk semua ambang** antara 180 sampai 300. Artinya ada jurang lebar antara data normal (maks. HR 169,2 dan SBP 220) dan data ekstrem (min. HR 314,6 dan SBP 381,8), sehingga angka batas tidak menentukan hasil.
- Ambang 220 (HR) dan 250 (SBP) dipilih dari batas fisiologis yang wajar; hasilnya terkonfirmasi oleh jurang di data.
- Selanjutnya cek apakah kolom lain pada baris itu juga rusak.

In [ ]:
grp = extreme_hr & extreme_sbp
cmp = pd.concat({"Kelompok ekstrem (92 baris)": df.loc[grp, ["glucose", "creatinine"]].describe().T[["mean", "50%", "max"]],
                 "Baris lainnya": df.loc[~grp, ["glucose", "creatinine"]].describe().T[["mean", "50%", "max"]]}, axis=1).round(1)
print(cmp)
print("\nSurvival rate kelompok ekstrem:", round(df.loc[grp, "survived_icu"].mean(), 3),
      "| baris lainnya:", round(df.loc[~grp, "survived_icu"].mean(), 3))

**Kesimpulan:** 92 baris itu rusak **satu blok** (HR, SBP, glucose, dan creatinine sama-sama ekstrem), bukan pasien yang memang parah. Survival rate-nya sama dengan baris lain, jadi membuangnya tidak membiaskan target.

> **Poin presentasi:** noise tidak acak: 92 baris (2%) rusak bersamaan pada 4 kolom. Ini mencerminkan *Garbage in, garbage out*.

### 3.4 Inkonsistensi Antar Atribut

In [ ]:
bp_lt = (df["systolic_bp"] < df["diastolic_bp"]).sum()
bp_eq = (df["systolic_bp"] == df["diastolic_bp"]).sum()
print("Sistolik < diastolik  :", bp_lt, "baris (tidak mungkin secara fisiologis)")
print("Sistolik = diastolik  :", bp_eq, "baris (selisih nol juga tidak wajar)")
print("oxygen_saturation tepat 100 (batas alat ukur / capping):", (df["oxygen_saturation"] == 100).sum(), "baris")

### 3.5 Distribusi Target

In [ ]:
target_count = df["survived_icu"].value_counts().sort_index()
target_pct = (df["survived_icu"].value_counts(normalize=True).sort_index() * 100).round(2)
print(pd.DataFrame({"jumlah": target_count, "persen": target_pct}))

plt.figure(figsize=(5, 4))
bars = plt.bar(["0 (tidak selamat)", "1 (selamat)"], target_count.values, color=["tab:red", "tab:green"])
for b, v, p in zip(bars, target_count.values, target_pct.values):
    plt.text(b.get_x() + b.get_width() / 2, b.get_height(), f"{v}\n({p}%)", ha="center", va="bottom")
plt.title("Distribusi target survived_icu"); plt.ylabel("Jumlah pasien")
plt.tight_layout(); plt.show()

Kelas tidak seimbang (64% selamat). **Menebak "selamat" untuk semua pasien sudah benar 64% dari waktu**, jadi accuracy model harus dibandingkan dengan angka ini, bukan dengan 50%.

### Ringkasan masalah dan rencana penanganan

| No | Masalah | Jenis (slide) | Penanganan | Bagian |
|----|---------|---------------|------------|--------|
| 1 | Duplikat tersembunyi (persis dan hampir persis) | Duplicate | Deteksi, buang salinan | 4.1 |
| 2 | 92 baris rusak satu blok | Noise / Accuracy | Buang baris | 4.2 |
| 3 | Sistolik <= diastolik | Consistency | Uji: tukar vs NaN vs biarkan | 6.1 |
| 4 | Missing value berkelompok (MCAR) | Completeness | Bandingkan deletion, ignoring, imputasi | 6.2 |
| 5 | Outlier sisa, distribusi miring | Outlier | Winsorizing, log, dsb. | 6.3 |
| 6 | Skala beda-beda | Transformation | Scaling | 7.2 |
| 7 | Fitur kurang informatif | Feature engineering / reduction | Fitur turunan, seleksi, PCA | 7.6, 8 |

## 4. Cleaning Tahap 1: Aturan Pasti (sebelum Split)

Langkah di tahap ini **tidak menghitung statistik apa pun dari data** (hanya membuang baris yang jelas rusak atau kembar), sehingga aman dilakukan sebelum split dan tidak menimbulkan *data leakage*. Justru, duplikat **harus** dibuang sebelum split supaya salinan tidak jatuh ke train sementara aslinya ke test.

### 4.1 Removing Duplicate Data

Contoh dari slide dulu: `drop_duplicates(subset=..., keep=...)`.

In [ ]:
toy = pd.DataFrame({"brand": ["Yum Yum", "Yum Yum", "Indomie", "Indomie", "Indomie"],
                    "style": ["cup", "cup", "cup", "pack", "pack"],
                    "rating": [4, 4, 3.5, 15, 5]})
print("drop_duplicates():\n", toy.drop_duplicates())
print("\nsubset=['brand']:\n", toy.drop_duplicates(subset=["brand"]))
print("\nsubset=['brand','style'], keep='last':\n", toy.drop_duplicates(subset=["brand", "style"], keep="last"))

`drop_duplicates` hanya menangkap baris yang **sama persis**. Di dataset ini ada juga *near-duplicate*: salinan yang sama di semua kolom yang terisi, tetapi salah satunya punya kolom kosong. Bagi `drop_duplicates`, `NaN` berbeda dengan angka, sehingga pasangan seperti ini lolos.

Fungsi di bawah mencari pasangan baris yang **semua kolom terisi di kedua baris bernilai sama** (dan minimal 9 kolom cocok, supaya kebetulan tidak terhitung). Dari tiap pasangan, baris yang lebih tidak lengkap dibuang.

In [ ]:
def find_duplicate_rows(data, feature_cols, min_match=9):
    """Mengembalikan (indeks_baris_dibuang, pasangan).
    Dua baris dianggap kembar bila pada kolom yang terisi di KEDUANYA tidak ada selisih
    dan minimal `min_match` kolom cocok. Baris yang lebih banyak NaN dibuang;
    jika seri, dibuang yang berawalan ICU_DUP_ (atau yang muncul belakangan)."""
    A = data[feature_cols].to_numpy(dtype=float)
    n_nan = np.isnan(A).sum(axis=1)
    is_dup_id = data["patient_id"].str.startswith("ICU_DUP_").to_numpy()
    to_drop, pairs = set(), []
    for i in range(len(A) - 1):
        a, B = A[i], A[i + 1:]
        both = ~np.isnan(a) & ~np.isnan(B)
        mismatch = ((a != B) & both).sum(axis=1)
        match = ((a == B) & both).sum(axis=1)
        for k in np.where((mismatch == 0) & (match >= min_match))[0]:
            j = i + 1 + k
            pairs.append((i, j))
            if n_nan[i] != n_nan[j]:
                drop = i if n_nan[i] > n_nan[j] else j
            else:
                drop = i if is_dup_id[i] else j
            to_drop.add(drop)
    return sorted(data.index[list(to_drop)]), pairs

dup_idx, dup_pairs = find_duplicate_rows(df, FEATURES)

exact = df.duplicated(subset=FEATURES + ["survived_icu"], keep="first").sum()
print("Pasangan kembar ditemukan  :", len(dup_pairs))
print("  - persis sama            :", exact)
print("  - hampir sama (beda NaN) :", len(dup_idx) - exact)
print("Baris yang dibuang         :", len(dup_idx))

In [ ]:
# Validasi: apakah temuan ini masuk akal?
pos = {idx: k for k, idx in enumerate(df.index)}
all_ok_target = all(df["survived_icu"].iloc[i] == df["survived_icu"].iloc[j] for i, j in dup_pairs)
ids_per_pair = [(df["patient_id"].iloc[i].startswith("ICU_DUP_")) + (df["patient_id"].iloc[j].startswith("ICU_DUP_"))
                for i, j in dup_pairs]

print("Target sama pada semua pasangan          :", all_ok_target)
print("Pasangan dengan tepat 1 ID berawalan DUP :", sum(x == 1 for x in ids_per_pair), "dari", len(dup_pairs))
print("Tidak ada pasangan yang kedua barisnya terbuang:",
      not any((df.index[i] in set(dup_idx)) and (df.index[j] in set(dup_idx)) for i, j in dup_pairs))

sisa_dup_id = df["patient_id"].str.startswith("ICU_DUP_").sum() - len(dup_pairs)
print("\nID berawalan ICU_DUP_ yang tidak ditemukan pasangannya:", sisa_dup_id,
      "(tidak dibuang karena tidak ada bukti duplikat)")

**Hasil:** 77 pasangan kembar (58 persis sama, 19 hampir sama). Pada setiap pasangan tepat satu ID berawalan `ICU_DUP_` dan targetnya sama, jadi deteksi berbasis isi data ini **cocok dengan petunjuk pola ID** tanpa perlu memakai ID-nya. Ini penting karena aturan berbasis isi tetap bisa dipakai pada data baru yang tidak punya pola ID seperti itu.

Tiga ID `ICU_DUP_` sisanya tidak punya pasangan di mana pun, jadi dibiarkan.

### 4.2 Membuang Baris Rusak (Noise Ekstrem)

Dari bagian 3.3: 92 baris dengan HR > 220 **dan** SBP > 250. Baris dibuang seluruhnya (bukan hanya nilai ekstremnya) karena korupsinya satu blok: kolom lain pada baris itu (glucose, creatinine) ikut rusak, sehingga memperbaiki sebagian saja tidak aman.

In [ ]:
corrupt_idx = df.index[(df["heart_rate"] > 220) & (df["systolic_bp"] > 250)]
print("Baris rusak:", len(corrupt_idx))
print("Tumpang tindih dengan duplikat:", len(set(corrupt_idx) & set(dup_idx)))

removed_idx = sorted(set(dup_idx) | set(corrupt_idx))
df_c = df.drop(index=removed_idx)          # indeks asli dipertahankan
print("\nUkuran data:", df.shape, "->", df_c.shape)

### 4.3 Validasi Hasil Tahap 1

In [ ]:
cleaning_log = pd.DataFrame({
    "proses": ["Data awal", "Duplikat persis", "Duplikat hampir sama (beda NaN)", "Baris rusak (HR > 220 dan SBP > 250)", "Data akhir tahap 1"],
    "jumlah": [len(df), exact, len(dup_idx) - exact, len(corrupt_idx), len(df_c)],
    "jenis": ["baris", "dibuang", "dibuang", "dibuang", "baris"]})
display(cleaning_log)
print(f"Total dibuang: {len(removed_idx)} baris ({100 * len(removed_idx) / len(df):.2f}%)")

# Pemeriksaan otomatis: kalau salah satu gagal, notebook berhenti dengan error
assert df_c["patient_id"].is_unique
assert not df_c.duplicated(subset=FEATURES + ["survived_icu"]).any()
assert not ((df_c["heart_rate"] > 220) & (df_c["systolic_bp"] > 250)).any()
violations = {c: int(((df_c[c] < lo) | (df_c[c] > hi)).sum()) for c, (lo, hi) in RANGES.items() if c != "bmi"}
assert sum(violations.values()) == 0, violations
assert set(df_c["survived_icu"].unique()) <= {0, 1}
print("Semua pemeriksaan lolos: tidak ada nilai di luar rentang fisiologis (selain BMI, ditangani di 6.3).")

In [ ]:
# Statistik sebelum vs sesudah
cols = ["heart_rate", "systolic_bp", "glucose", "creatinine"]
pd.concat({"Sebelum": df[cols].agg(["mean", "std", "max"]).T,
           "Sesudah": df_c[cols].agg(["mean", "std", "max"]).T}, axis=1).round(2)

### 4.4 Korelasi Setelah Cleaning (Spearman)

Spearman memakai peringkat, bukan nilai asli, sehingga lebih tahan terhadap outlier. Nilai -1 sampai +1: +1 naik bersama, -1 berlawanan, 0 tidak berhubungan.

In [ ]:
num_df = df_c.drop(columns="patient_id")
corr = num_df.corr(method="spearman")

fig, ax = plt.subplots(figsize=(9, 7.5))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=90)
ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns)
plt.colorbar(im); plt.title("Korelasi Spearman"); plt.tight_layout(); plt.show()

(corr["survived_icu"].drop("survived_icu")
     .sort_values(key=abs, ascending=False).round(3).to_frame("Spearman dengan target"))

**Temuan penting untuk seluruh tugas:** hanya **empat** kolom yang berhubungan dengan target secara berarti: `age` (-0,21), `severity_score` (-0,15), `ventilation_required` (-0,06), dan `chronic_conditions` (-0,06). Semua kolom tanda vital dan lab (termasuk yang paling kotor) berkorelasi |r| < 0,04 dengan target.

> **Poin presentasi:** kolom yang paling kotor justru yang paling tidak informatif untuk prediksi. Ini menjelaskan hasil akhir nanti: membersihkannya membuat data valid, tetapi tidak menaikkan accuracy secara besar.

## 5. Split Data dan Alat Bantu

Mulai dari sini, semua teknik yang **belajar dari data** (imputasi, scaling, winsorizing, binning, seleksi fitur, PCA) hanya boleh dipelajari dari **data latih**. Karena itu data dibagi dulu: 80% latih, 20% uji. Test set **tidak disentuh** sampai evaluasi akhir (bagian 9); semua perbandingan teknik memakai 5-fold cross-validation pada data latih.

In [ ]:
X = df_c[FEATURES]
y = df_c["survived_icu"]

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
print("Latih:", X_tr.shape, "| Uji:", X_te.shape)
print("Accuracy menebak kelas mayoritas pada test:", round(y_te.value_counts(normalize=True).max(), 4))

**(utilitas)** Fungsi evaluasi dan transformer buatan. Satu model (Logistic Regression) dipakai di semua eksperimen.

In [ ]:
EXPERIMENT_LOG = []

def log_result(tahap, variasi, mean, std):
    EXPERIMENT_LOG.append({"Tahap": tahap, "Variasi": variasi,
                           "CV accuracy": round(mean, 4), "Std": round(std, 4)})

def lr_model():
    return LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)

def cv_acc(make_pipe, Xd, yd, row_filter=None, val_filter=None, n_splits=5):
    """5-fold stratified CV. make_pipe: fungsi yang membuat pipeline baru tiap fold.
    row_filter(X_trainfold) -> mask baris latih yang dipakai (dihitung hanya dari train fold).
    val_filter(X_valfold)   -> mask baris validasi yang dinilai (untuk listwise deletion)."""
    skf = StratifiedKFold(n_splits, shuffle=True, random_state=RANDOM_STATE)
    scores = []
    for tr, va in skf.split(Xd, yd):
        Xtr, ytr, Xva, yva = Xd.iloc[tr], yd.iloc[tr], Xd.iloc[va], yd.iloc[va]
        if row_filter is not None:
            keep = row_filter(Xtr); Xtr, ytr = Xtr[keep], ytr[keep]
        if val_filter is not None:
            keep = val_filter(Xva); Xva, yva = Xva[keep], yva[keep]
        pipe = make_pipe(); pipe.fit(Xtr, ytr)
        scores.append(accuracy_score(yva, pipe.predict(Xva)))
    return float(np.mean(scores)), float(np.std(scores))

In [ ]:
SKEWED = ["creatinine", "wbc_count"]   # kolom dengan skewness > 1 (lihat tabel di 6.3)

class Winsorizer(BaseEstimator, TransformerMixin):
    """Memotong tiap kolom ke persentil bawah/atas yang dihitung dari data latih."""
    def __init__(self, lower=1, upper=99):
        self.lower, self.upper = lower, upper
    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        self.lo_ = np.nanpercentile(X, self.lower, axis=0)
        self.hi_ = np.nanpercentile(X, self.upper, axis=0)
        return self
    def transform(self, X):
        return np.clip(np.asarray(X, dtype=float), self.lo_, self.hi_)

class IQRClipper(BaseEstimator, TransformerMixin):
    """Memotong ke [Q1 - k*IQR, Q3 + k*IQR] dari data latih."""
    def __init__(self, k=1.5):
        self.k = k
    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        q1, q3 = np.nanpercentile(X, 25, axis=0), np.nanpercentile(X, 75, axis=0)
        self.lo_, self.hi_ = q1 - self.k * (q3 - q1), q3 + self.k * (q3 - q1)
        return self
    def transform(self, X):
        return np.clip(np.asarray(X, dtype=float), self.lo_, self.hi_)

class DecimalScaler(BaseEstimator, TransformerMixin):
    """Decimal scaling: v' = v / 10^j, j = bilangan bulat terkecil sehingga max(|v'|) < 1."""
    def fit(self, X, y=None):
        m = np.nanmax(np.abs(np.asarray(X, dtype=float)), axis=0)
        self.j_ = np.where(m > 0, np.floor(np.log10(m)) + 1, 0)
        return self
    def transform(self, X):
        return np.asarray(X, dtype=float) / (10.0 ** self.j_)

class EntropyBinner(BaseEstimator, TransformerMixin):
    """Diskretisasi supervised: batas bin = threshold pohon keputusan (criterion='entropy')."""
    def __init__(self, max_bins=5, min_samples_leaf=100):
        self.max_bins, self.min_samples_leaf = max_bins, min_samples_leaf
    def fit(self, X, y):
        X = np.asarray(X, dtype=float); y = np.asarray(y)
        self.edges_ = []
        for j in range(X.shape[1]):
            col = X[:, j]; ok = ~np.isnan(col)
            tree = DecisionTreeClassifier(criterion="entropy", max_leaf_nodes=self.max_bins,
                                          min_samples_leaf=self.min_samples_leaf, random_state=0)
            tree.fit(col[ok].reshape(-1, 1), y[ok])
            self.edges_.append(np.sort(tree.tree_.threshold[tree.tree_.feature >= 0]))
        return self
    def transform(self, X):
        X = np.asarray(X, dtype=float)
        return np.hstack([np.eye(len(e) + 1)[np.digitize(X[:, j], e)] for j, e in enumerate(self.edges_)])

In [ ]:
# ---- Fungsi per-baris (tidak belajar dari data) ----
PHYS_RANGES = {c: r for c, r in RANGES.items() if c != "bmi"}

def fix_values(data, bp_mode="nan"):
    """Perbaikan nilai per baris.
    1) Nilai di luar rentang fisiologis -> NaN (jaring pengaman untuk data baru).
    2) Tekanan darah tidak konsisten (sistolik <= diastolik) ditangani sesuai bp_mode:
       'nan' = keduanya NaN | 'swap' = ditukar | 'none' = dibiarkan."""
    d = data.copy()
    for c, (lo, hi) in PHYS_RANGES.items():
        d.loc[(d[c] < lo) | (d[c] > hi), c] = np.nan
    if bp_mode == "nan":
        bad = d["systolic_bp"] <= d["diastolic_bp"]
        d.loc[bad, ["systolic_bp", "diastolic_bp"]] = np.nan
    elif bp_mode == "swap":
        bad = d["systolic_bp"] < d["diastolic_bp"]
        old = d.loc[bad, "systolic_bp"].copy()
        d.loc[bad, "systolic_bp"] = d.loc[bad, "diastolic_bp"].values
        d.loc[bad, "diastolic_bp"] = old.values
    return d

def add_missing_flags(d):
    """Indikator missing: n_missing = jumlah pengukuran yang tidak tersedia pada pasien itu.
    (Lima indikator per-kolom digabung jadi satu karena berkorelasi 0,8-0,99; lihat bagian 6.2.)"""
    d = d.copy()
    d["n_missing"] = d[MEASURED].isna().sum(axis=1)
    return d

def add_extraction(d):
    d = d.copy()
    d["pulse_pressure"] = d["systolic_bp"] - d["diastolic_bp"]
    d["map"] = (d["systolic_bp"] + 2 * d["diastolic_bp"]) / 3
    d["shock_index"] = d["heart_rate"] / d["systolic_bp"]
    return d

def add_combination(d):
    d = d.copy()
    d["age_x_severity"] = d["age"] * d["severity_score"]
    d["vent_x_severity"] = d["ventilation_required"] * d["severity_score"]
    return d

def make_preprocessor(imputer=None, scaler=None, outlier="winsor", log=True):
    steps = []
    if log:
        steps.append(("log", ColumnTransformer(
            [("log", FunctionTransformer(np.log1p), SKEWED)], remainder="passthrough")))
    if outlier == "winsor":
        steps.append(("outlier", Winsorizer(1, 99)))
    elif outlier == "iqr":
        steps.append(("outlier", IQRClipper(1.5)))
    steps.append(("scale", scaler if scaler is not None else StandardScaler()))
    steps.append(("impute", imputer if imputer is not None else KNNImputer(n_neighbors=5)))
    return Pipeline(steps)

def full_pipe(imputer=None, flags=False, extract=False, combine=False, **prep_kwargs):
    steps = []
    if extract: steps.append(("extract", FunctionTransformer(add_extraction)))
    if combine: steps.append(("combine", FunctionTransformer(add_combination)))
    if flags:   steps.append(("flags", FunctionTransformer(add_missing_flags)))
    steps.append(("prep", make_preprocessor(imputer=imputer, **prep_kwargs)))
    steps.append(("model", lr_model()))
    return Pipeline(steps)

### 5.1 Eksperimen kecil: apakah baris kotor benar-benar merusak model?

Pertanyaan wajar dari audiens: *"Kalau baris rusak dan duplikat dibuang, apakah modelnya jadi lebih baik?"* Dicoba langsung: latih model dengan data bersih, lalu tambahkan baris yang dibuang ke data latih, dan uji **selalu pada pasien bersih yang sama**. Diulang 20 kali dengan pembagian acak berbeda supaya tidak kebetulan.

In [ ]:
removed_rows = df.loc[removed_idx]
corrupt_rows = df.loc[corrupt_idx]
dup_rows = df.loc[dup_idx]

def quick_lr():
    return Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler()), ("m", lr_model())])

res = {"Bersih": [], "+ baris rusak": [], "+ duplikat": [], "+ keduanya (data mentah)": []}
for seed in range(20):
    Xa, Xb, ya, yb = train_test_split(X, y, test_size=0.2, stratify=y, random_state=seed)
    for name, extra in [("Bersih", []), ("+ baris rusak", [corrupt_rows]), ("+ duplikat", [dup_rows]),
                        ("+ keduanya (data mentah)", [corrupt_rows, dup_rows])]:
        Xt = pd.concat([Xa] + [e[FEATURES] for e in extra]); yt = pd.concat([ya] + [e["survived_icu"] for e in extra])
        res[name].append(accuracy_score(yb, quick_lr().fit(Xt, yt).predict(Xb)))

summary_dirty = pd.DataFrame({k: {"rata-rata accuracy": np.mean(v), "std antar-split": np.std(v)} for k, v in res.items()}).T.round(4)
summary_dirty

**Hasil:** accuracy hampir identik di keempat kondisi (selisih jauh di bawah simpangan baku antar-split). Untuk Logistic Regression dengan fitur seperti ini, baris rusak dan duplikat **tidak mengubah accuracy secara terukur**.

Ini bukan alasan untuk melewatkan cleaning. Alasan membersihkan data tetap kuat: (1) validitas data (denyut 1000/menit tidak mungkin ada), (2) duplikat yang jatuh ke train *dan* test membuat evaluasi terlalu optimis, dan (3) statistik deskriptif/laporan jadi benar. Tetapi kita harus jujur bahwa efek langsungnya ke accuracy kecil.

> **Poin presentasi:** nilai cleaning ada pada **kebenaran data**, bukan lonjakan accuracy. Menunjukkan eksperimen ini justru memperkuat kredibilitas.

## 6. Cleaning Tahap 2: Diuji pada Data Latih

Keputusan di tahap ini tidak jelas dari logika saja, jadi dibandingkan secara empiris dengan 5-fold CV pada data latih (Logistic Regression, accuracy).

### 6.1 Tekanan Darah Tidak Konsisten (Sistolik <= Diastolik)

Ada 345 baris di data bersih. Tiga kemungkinan perlakuan:
1. **Tukar** nilai sistolik dan diastolik (asumsi: salah input terbalik)
2. **NaN** keduanya, lalu diimputasi (asumsi: salah satu salah, tidak tahu yang mana)
3. **Biarkan** apa adanya

In [ ]:
bad = X_tr["systolic_bp"] <= X_tr["diastolic_bp"]
print("Baris tidak konsisten di data latih:", bad.sum())

# Apakah asumsi 'tertukar' masuk akal? Bandingkan dengan baris normal
swapped = X_tr.loc[bad]
normal = X_tr.loc[~bad]
print("\nSetelah ditukar, pada baris ini: sistolik median =", swapped["diastolic_bp"].median().round(1),
      "| selisih sistolik-diastolik median =", (swapped["diastolic_bp"] - swapped["systolic_bp"]).median().round(1))
print("Pada baris normal              : sistolik median =", normal["systolic_bp"].median().round(1),
      "| selisih sistolik-diastolik median =", (normal["systolic_bp"] - normal["diastolic_bp"]).median().round(1))

In [ ]:
rows = []
for mode, label in [("none", "Biarkan"), ("swap", "Tukar"), ("nan", "Jadikan NaN lalu imputasi")]:
    Xf = fix_values(X_tr, bp_mode=mode)
    m, s = cv_acc(lambda: full_pipe(imputer=KNNImputer(n_neighbors=5)), Xf, y_tr)
    rows.append({"Perlakuan": label, "CV accuracy": round(m, 4), "Std": round(s, 4)})
    log_result("6.1 Tekanan darah", label, m, s)
pd.DataFrame(rows).set_index("Perlakuan")

**Keputusan: jadikan NaN lalu imputasi.**
- Setelah ditukar, baris-baris itu **masih tidak khas**: selisih sistolik-diastolik hanya sekitar 12 mmHg (normal sekitar 48), jadi dugaan "hanya tertukar" kurang terbukti. Kita tidak tahu nilai mana yang benar.
- Ketiga perlakuan memberi accuracy yang setara, jadi pilih yang paling jujur terhadap ketidakpastian: tidak mengarang nilai (menukar), tetapi mengisinya dari pasien yang mirip.
- Ini sebuah **asumsi** dan sebaiknya disebut sebagai asumsi dalam laporan.

In [ ]:
# Terapkan keputusan ke data latih dan data uji (aturan per-baris, tidak belajar dari data)
X_tr_f = fix_values(X_tr, bp_mode="nan")
X_te_f = fix_values(X_te, bp_mode="nan")
print("NaN baru pada tekanan darah di data latih:",
      int(X_tr_f[["systolic_bp", "diastolic_bp"]].isna().sum().sum() - X_tr[["systolic_bp", "diastolic_bp"]].isna().sum().sum()))

### 6.2 Handling Missing Value

Semua perlakuan dari slide dibandingkan: **deletion** (listwise, pairwise, hapus kolom), **ignoring**, dan **imputasi**. Pipeline selalu menghitung statistik imputasi hanya dari train fold.

In [ ]:
# Deletion: biaya kehilangan data
n_all = len(X_tr_f); n_complete = X_tr_f.dropna().shape[0]
print(f"Listwise deletion: {n_all} -> {n_complete} baris ({100*(1-n_complete/n_all):.1f}% data hilang)")

pair = X_tr_f[["bmi", "age"]]
print("\nPairwise: korelasi bmi-age memakai", pair.dropna().shape[0], "baris pasangan (pandas .corr() otomatis pairwise)")
print("  korelasi pairwise :", round(pair.corr().iloc[0, 1], 4))
print("  korelasi listwise :", round(X_tr_f.dropna()[["bmi", "age"]].corr().iloc[0, 1], 4))

miss_pct = X_tr_f.isna().mean()
cols_to_drop = miss_pct[miss_pct > 0.10].index.tolist()
print("\nKolom dengan missing > 10% (kandidat dihapus):", cols_to_drop)

In [ ]:
# Imputasi di pandas seperti di slide (demo; ffill/bfill hanya masuk akal untuk data berurutan/time series)
demo = X_tr_f[["bmi", "oxygen_saturation"]].head(6)
print("Asli:\n", demo)
print("\nfillna(0):\n", demo.fillna(0))
print("\nfillna(mean):\n", demo.fillna(demo.mean()).round(2))
print("\nffill:\n", demo.ffill())
print("\nbfill:\n", demo.bfill())

In [ ]:
results = []
def add(variasi, mean, std, note=""):
    results.append({"Perlakuan missing": variasi, "CV accuracy": round(mean, 4), "Std": round(std, 4), "Catatan": note})
    log_result("6.2 Missing value", variasi, mean, std)

# --- DELETION ---
def complete_rows(d): return d.notna().all(axis=1)
m, s = cv_acc(lambda: Pipeline([("scale", StandardScaler()), ("model", lr_model())]),
              X_tr_f, y_tr, row_filter=complete_rows, val_filter=complete_rows)
add("Listwise deletion", m, s, "dinilai hanya pada baris lengkap (tidak setara)")

def drop_cols(d): return d.drop(columns=cols_to_drop)
m, s = cv_acc(lambda: Pipeline([("drop", FunctionTransformer(drop_cols)), ("impute", SimpleImputer(strategy="median")),
                                ("scale", StandardScaler()), ("model", lr_model())]), X_tr_f, y_tr)
add("Hapus kolom missing > 10%", m, s, "sisa NaN diisi median")

# --- IGNORING: algoritma pohon yang menangani NaN sendiri (model lain, hanya untuk peragaan) ---
m, s = cv_acc(lambda: Pipeline([("model", HistGradientBoostingClassifier(random_state=RANDOM_STATE))]), X_tr_f, y_tr)
add("Ignoring (pohon, NaN native)", m, s, "bukan Logistic Regression")

# --- IMPUTATION ---
imputers = {
    "Imputasi konstanta 0": SimpleImputer(strategy="constant", fill_value=0),
    "Imputasi mean":        SimpleImputer(strategy="mean"),
    "Imputasi median":      SimpleImputer(strategy="median"),
    "Imputasi modus":       SimpleImputer(strategy="most_frequent"),
    "KNNImputer (k=5)":     KNNImputer(n_neighbors=5),
    "IterativeImputer":     IterativeImputer(max_iter=10, random_state=RANDOM_STATE),
}
for name, imp in imputers.items():
    m, s = cv_acc(lambda imp=imp: full_pipe(imputer=imp), X_tr_f, y_tr)
    add(name, m, s)

m, s = cv_acc(lambda: full_pipe(imputer=KNNImputer(n_neighbors=5), flags=True), X_tr_f, y_tr)
add("KNNImputer + indikator missing", m, s, "tambah kolom n_missing")

pd.DataFrame(results).set_index("Perlakuan missing")

In [ ]:
# Mengapa hanya satu indikator missing (n_missing), bukan satu per kolom?
flags = pd.DataFrame({c: X_tr_f[c].isna().astype(int) for c in MISS_COLS})
flags["n_missing"] = X_tr_f[MEASURED].isna().sum(axis=1)
flags.corr().round(2)

Lima indikator per kolom berkorelasi **0,8 sampai 0,99** satu sama lain (kolomnya kosong bersamaan), jadi hampir tidak ada informasi tambahan, dan pada model linear mereka saling meniadakan sehingga koefisiennya tidak bisa ditafsirkan. Karena itu cukup **satu** indikator: `n_missing` (jumlah pengukuran yang tidak tersedia).

**Cara membaca:** selisih antar perlakuan lebih kecil daripada simpangan baku antar-fold (kolom Std), jadi **tidak ada pemenang yang pasti**. Itu wajar: missing-nya MCAR dan kolom yang kosong hampir tidak berkorelasi dengan target.

**Keputusan: KNNImputer + satu indikator missing (`n_missing`).** Alasan non-accuracy:
- KNNImputer mengisi dari pasien yang mirip (memanfaatkan hubungan antar fitur) dan tidak mengerutkan variasi seperti mean/median.
- Indikator `n_missing` menyimpan informasi "data ini tidak tercatat" tanpa menambah banyak kolom.
- Listwise deletion tidak dipilih karena membuang seperempat lebih data latih.

> **Poin presentasi:** 10 perlakuan missing diuji; hasilnya setara, jadi keputusan didasarkan pada kelayakan metode, bukan selisih accuracy yang tidak signifikan.

### 6.3 Handling Outliers

Noise mustahil sudah dibuang di bagian 4. Masih ada nilai ekstrem yang secara statistik menyimpang (mis. glucose tinggi, BMI 9). Deteksi dengan **IQR** dan **z-score**, lalu bandingkan beberapa penanganan.

In [ ]:
rows = []
for c in NUM_COLS:
    col = X_tr_f[c].dropna()
    q1, q3 = col.quantile([0.25, 0.75]); iqr = q3 - q1
    z = (col - col.mean()) / col.std()
    rows.append({"kolom": c,
                 "outlier_IQR (1.5xIQR)": int(((col < q1 - 1.5*iqr) | (col > q3 + 1.5*iqr)).sum()),
                 "outlier_zscore (|z|>3)": int((z.abs() > 3).sum()),
                 "skewness": round(col.skew(), 2)})
pd.DataFrame(rows).set_index("kolom")

In [ ]:
def zscore_filter(d, thr=3):
    sub = d[NUM_COLS]
    z = (sub - sub.mean()) / sub.std()
    return ((z.abs() <= thr) | z.isna()).all(axis=1)

variants = {
    "Tidak ditangani":               dict(kw=dict(outlier=None, log=False)),
    "Hapus baris |z|>3":             dict(kw=dict(outlier=None, log=False), rf=zscore_filter),
    "Clipping IQR (1.5xIQR)":        dict(kw=dict(outlier="iqr", log=False)),
    "Winsorizing 1%-99%":            dict(kw=dict(outlier="winsor", log=False)),
    "Log1p (kolom miring)":          dict(kw=dict(outlier=None, log=True)),
    "Log1p + Winsorizing":           dict(kw=dict(outlier="winsor", log=True)),
    "RobustScaler (tanpa clipping)": dict(kw=dict(outlier=None, log=False, scaler=RobustScaler())),
}
rows = []
for name, cfg in variants.items():
    m, s = cv_acc(lambda cfg=cfg: full_pipe(imputer=SimpleImputer(strategy="median"), **cfg["kw"]),
                  X_tr_f, y_tr, row_filter=cfg.get("rf"))
    rows.append({"Penanganan outlier": name, "CV accuracy": round(m, 4), "Std": round(s, 4)})
    log_result("6.3 Outlier", name, m, s)
pd.DataFrame(rows).set_index("Penanganan outlier")

**Keputusan: log1p + winsorizing 1%-99%.** Log hanya dipakai untuk kolom yang jelas miring: `creatinine` (skewness sekitar 2,0) dan `wbc_count` (sekitar 1,6). `glucose` sudah simetris (0,23) setelah baris rusak dibuang, jadi tidak perlu di-log. Winsorizing memotong sisa nilai ekstrem **tanpa membuang baris** (menghapus baris berdasarkan z-score membuang data yang mungkin valid). Seperti sebelumnya, selisih accuracy antar variasi kecil.

## 7. Data Transformation

### 7.1 Encoding Data Kategorik

Dataset ini hampir seluruhnya numerik. `gender` dan `ventilation_required` sudah 0/1. Agar tiga teknik dari slide tetap dapat diperagakan, dibuat satu kolom kategorik dari BMI memakai kategori WHO (*attribute construction*).

In [ ]:
bmi_labels = ["underweight", "normal", "overweight", "obese"]
bmi_cat = pd.cut(X_tr_f["bmi"], bins=[0, 18.5, 25, 30, 100], labels=bmi_labels, right=False)
demo = pd.DataFrame({"bmi": X_tr_f["bmi"], "bmi_category": bmi_cat}).dropna().head(8)
print(demo)

# 1) Label Encoder: label -> 0..n-1 (urut alfabet, TANPA makna urutan; cocok untuk target)
le = LabelEncoder().fit(demo["bmi_category"].astype(str))
print("\nLabelEncoder:", dict(zip(le.classes_, le.transform(le.classes_))))

# 2) Ordinal Encoder: urutan ditentukan manual (untuk atribut ORDINAL)
oe = OrdinalEncoder(categories=[bmi_labels])
print("OrdinalEncoder (urutan manual):", oe.fit_transform(demo[["bmi_category"]].astype(str)).ravel())

# 3) One-hot Encoder: tiap kategori jadi kolom 0/1 (untuk atribut NOMINAL)
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
oh = ohe.fit_transform(demo[["bmi_category"]].astype(str))
print("\nOneHotEncoder:")
print(pd.DataFrame(oh, columns=ohe.get_feature_names_out(["bmi_category"]), index=demo.index))

**Kapan memakai apa:** one-hot untuk nominal (tidak ada urutan), ordinal encoder untuk ordinal (ada urutan), label encoder terutama untuk target. `gender` dan `ventilation_required` sudah biner 0/1 sehingga tidak perlu diubah.

### 7.2 Feature Scaling

Tanpa scaling, fitur berskala besar (glucose ratusan, usia puluhan, creatinine satuan) mendominasi model berbasis jarak atau gradien. Teknik dari slide: **min-max**, **z-score**, **unit conversion**, **decimal scaling**; ditambah RobustScaler, PowerTransformer, QuantileTransformer.

In [ ]:
# Unit conversion: mg/dL -> satuan SI
print(pd.DataFrame({
    "glucose mg/dL": X_tr_f["glucose"].head(4), "glucose mmol/L": (X_tr_f["glucose"] / 18.016).head(4).round(2),
    "creatinine mg/dL": X_tr_f["creatinine"].head(4), "creatinine umol/L": (X_tr_f["creatinine"] * 88.4).head(4).round(1)}))

# Peragaan rumus pada kolom age
a = X_tr_f[["age"]]
print("\n", pd.DataFrame({
    "asli": a["age"].head(5),
    "min-max":  MinMaxScaler().fit_transform(a)[:5, 0].round(3),
    "z-score":  StandardScaler().fit_transform(a)[:5, 0].round(3),
    "decimal":  DecimalScaler().fit_transform(a)[:5, 0].round(3)}))
print("\nCek manual min-max:", round((a['age'].iloc[0]-a['age'].min())/(a['age'].max()-a['age'].min()), 3),
      "| z-score:", round((a['age'].iloc[0]-a['age'].mean())/a['age'].std(ddof=0), 3))

In [ ]:
scalers = {
    "Tanpa scaling": "passthrough", "Min-Max": MinMaxScaler(), "Z-score (Standard)": StandardScaler(),
    "Decimal scaling": DecimalScaler(), "RobustScaler": RobustScaler(), "PowerTransformer": PowerTransformer(),
    "QuantileTransformer": QuantileTransformer(n_quantiles=200, output_distribution="normal", random_state=RANDOM_STATE),
}
def scale_pipe(scaler):
    return Pipeline([("impute", SimpleImputer(strategy="median")), ("winsor", Winsorizer(1, 99)),
                     ("scale", scaler), ("model", lr_model())])
rows = []
for name, sc in scalers.items():
    m, s = cv_acc(lambda sc=sc: scale_pipe(sc), X_tr_f, y_tr)
    rows.append({"Scaler": name, "CV accuracy": round(m, 4), "Std": round(s, 4)})
    log_result("7.2 Scaling", name, m, s)
pd.DataFrame(rows).set_index("Scaler")

**Pengamatan:** pada Logistic Regression, scaling hanya berpengaruh kecil terhadap accuracy karena solver-nya sudah cukup tangguh. Scaling jauh lebih penting untuk model berbasis jarak seperti KNN, dan tetap dibutuhkan di pipeline ini karena **KNNImputer** menghitung jarak antar pasien. Dipilih **z-score (StandardScaler)**.

### 7.3 Smoothing (binning)

Meredam *noise* kecil: bagi nilai ke bin berisi jumlah sama (equal-depth), lalu ganti tiap nilai dengan **rata-rata bin**-nya (*smoothing by bin means*).

In [ ]:
hr = X_tr_f["heart_rate"].dropna()
bins = pd.qcut(hr, q=10, duplicates="drop")
hr_smooth = hr.groupby(bins).transform("mean")
print(pd.DataFrame({"heart_rate asli": hr.head(6), "setelah smoothing": hr_smooth.head(6).round(1)}))
print(f"\nStd sebelum: {hr.std():.2f} | sesudah: {hr_smooth.std():.2f} | nilai unik: {hr.nunique()} -> {hr_smooth.nunique()}")

### 7.4 Aggregation

Menggabungkan banyak objek menjadi satu: pasien individual diringkas per kelompok usia / jumlah penyakit kronis. Tujuan: mengurangi jumlah objek, mengubah skala (individu -> kelompok), dan melihat pola yang lebih stabil.

In [ ]:
view = X_tr_f.assign(survived_icu=y_tr)
view["kelompok_usia"] = pd.cut(view["age"], bins=[17, 40, 55, 65, 75, 100], labels=["18-40", "41-55", "56-65", "66-75", ">75"])

agg_age = view.groupby("kelompok_usia", observed=True).agg(
    jumlah_pasien=("age", "size"), survival_rate=("survived_icu", "mean"),
    rata2_severity=("severity_score", "mean"), persen_ventilator=("ventilation_required", "mean")).round(3)
print(agg_age)

agg_chronic = view.groupby("chronic_conditions").agg(jumlah_pasien=("age", "size"), survival_rate=("survived_icu", "mean")).round(3)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
agg_age["survival_rate"].plot(kind="bar", ax=ax[0], color="tab:blue", rot=0)
ax[0].set_title("Survival rate per kelompok usia"); ax[0].set_ylim(0, 1)
agg_chronic["survival_rate"].plot(kind="bar", ax=ax[1], color="tab:orange", rot=0)
ax[1].set_title("Survival rate per jumlah penyakit kronis"); ax[1].set_ylim(0, 1)
plt.tight_layout(); plt.show()

**Temuan:** survival rate turun jelas seiring usia dan jumlah penyakit kronis. Pola ini sejalan dengan korelasi di bagian 4.4 dan menjadi dasar menjelaskan **mengapa** usia adalah fitur terpenting.

### 7.5 Discretization

Mengubah atribut kontinu menjadi interval. Empat metode dari slide pada `age`:
1. **Equal-width**: lebar interval sama
2. **Equal-depth (frequency)**: jumlah sampel per interval hampir sama
3. **Cluster analysis**: batas ditentukan k-means
4. **Entropy-based (supervised)**: batas dipilih agar tiap interval sebisa mungkin murni terhadap target

In [ ]:
age_tr = X_tr_f[["age"]]
edges, bin_index = {}, {}
for name, strat in [("Equal-width", "uniform"), ("Equal-depth", "quantile"), ("Cluster (k-means)", "kmeans")]:
    kb = KBinsDiscretizer(n_bins=5, encode="ordinal", strategy=strat, random_state=RANDOM_STATE)
    bin_index[name] = kb.fit_transform(age_tr).ravel().astype(int)
    edges[name] = np.round(kb.bin_edges_[0], 1)

eb = EntropyBinner(max_bins=5).fit(age_tr, y_tr)
edges["Entropy-based (supervised)"] = np.round(eb.edges_[0], 1)
bin_index["Entropy-based (supervised)"] = np.digitize(age_tr["age"].values, eb.edges_[0])

for name in edges:
    print(f"\n{name}\n  batas: {edges[name]}")
    tab = pd.DataFrame({"bin": bin_index[name], "y": y_tr.values}).groupby("bin").agg(
        jumlah=("y", "size"), survival_rate=("y", "mean")).round(3)
    print(tab.T.to_string())

In [ ]:
def disc_pipe(kind):
    if kind == "raw":
        step = ColumnTransformer([("age", "passthrough", ["age"])], remainder="passthrough")
    elif kind == "entropy":
        step = ColumnTransformer([("age", EntropyBinner(5), ["age"])], remainder="passthrough")
    else:
        step = ColumnTransformer([("age", KBinsDiscretizer(n_bins=5, encode="onehot-dense", strategy=kind,
                                                           random_state=RANDOM_STATE), ["age"])], remainder="passthrough")
    return Pipeline([("disc", step), ("impute", SimpleImputer(strategy="median")),
                     ("winsor", Winsorizer(1, 99)), ("scale", StandardScaler()), ("model", lr_model())])

rows = []
for label, kind in [("age asli (tanpa diskretisasi)", "raw"), ("Equal-width", "uniform"),
                    ("Equal-depth", "quantile"), ("Cluster (k-means)", "kmeans"), ("Entropy-based", "entropy")]:
    m, s = cv_acc(lambda kind=kind: disc_pipe(kind), X_tr_f, y_tr)
    rows.append({"Perlakuan age": label, "CV accuracy": round(m, 4), "Std": round(s, 4)})
    log_result("7.5 Diskretisasi", label, m, s)
pd.DataFrame(rows).set_index("Perlakuan age")

**Pengamatan:** diskretisasi membuang sebagian informasi (usia 61 dan 62 menjadi sama), dan hasilnya tidak lebih baik daripada `age` kontinu. Manfaat utamanya adalah **interpretabilitas** (tabel survival rate per bin) dan kebutuhan algoritma yang hanya menerima atribut kategorik. Karena itu `age` tetap dipakai kontinu di model akhir.

### 7.6 Feature Engineering

| Jenis (slide) | Penerapan |
|---|---|
| Feature extraction | `pulse_pressure` (sistolik - diastolik), `map`, `shock_index` (HR / sistolik) |
| Feature encoding | Bagian 7.1 dan indikator missing (6.2) |
| Feature scaling | Bagian 7.2 |
| Feature selection | Bagian 8 |
| Feature combination | `age_x_severity`, `vent_x_severity` (interaksi dua variabel) |

In [ ]:
fe_variants = {
    "Tanpa fitur baru":    dict(),
    "+ extraction":        dict(extract=True),
    "+ combination":       dict(combine=True),
    "+ indikator missing (n_missing)": dict(flags=True),
    "+ semuanya":          dict(extract=True, combine=True, flags=True),
}
rows = []
for name, kw in fe_variants.items():
    m, s = cv_acc(lambda kw=kw: full_pipe(imputer=KNNImputer(n_neighbors=5), **kw), X_tr_f, y_tr)
    rows.append({"Fitur": name, "CV accuracy": round(m, 4), "Std": round(s, 4)})
    log_result("7.6 Feature engineering", name, m, s)
pd.DataFrame(rows).set_index("Fitur")

**Pengamatan:** fitur interaksi `age_x_severity` memang membawa informasi (dicek di bagian 8), tetapi peningkatan accuracy kecil dan masih dalam rentang Std. Fitur turunan dari kolom yang berisik (`pulse_pressure`, `map`, `shock_index`) tidak membantu karena kolom asalnya hampir tidak prediktif.

**Keputusan:** pipeline akhir memakai **combination** (interaksi yang bermakna secara medis dan berkorelasi paling kuat dengan target) dan **indikator missing**, tanpa fitur extraction.

## 8. Data Reduction

### 8.1 Variable Selection

Tiga pendekatan: (1) **domain knowledge**: `patient_id` sudah dibuang; (2) **filter statistik**: korelasi, mutual information, variance threshold; (3) **berbasis model**: importance Random Forest (hanya sebagai alat bantu analisis, bukan model akhir).

In [ ]:
X_fe = add_missing_flags(add_combination(add_extraction(X_tr_f)))
X_fe_imp = pd.DataFrame(SimpleImputer(strategy="median").fit_transform(X_fe), columns=X_fe.columns, index=X_fe.index)
print("Jumlah fitur setelah feature engineering:", X_fe.shape[1])

corr_target = X_fe_imp.corrwith(y_tr).sort_values(key=np.abs, ascending=False).round(3)
print("\nKorelasi dengan target (5 teratas):\n", corr_target.head(5))

cm = X_fe_imp.corr().abs()
pairs = [(a, b, round(cm.loc[a, b], 2)) for i, a in enumerate(cm.columns) for b in cm.columns[i+1:] if cm.loc[a, b] > 0.8]
print("\nPasangan fitur redundan (|r| > 0.8):")
for p in pairs: print("  ", p)

In [ ]:
mi = pd.Series(mutual_info_classif(X_fe_imp, y_tr, random_state=RANDOM_STATE), index=X_fe_imp.columns)
rf_imp = pd.Series(RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)
                   .fit(X_fe_imp, y_tr).feature_importances_, index=X_fe_imp.columns)
ranking = pd.DataFrame({"mutual_info": mi.round(4), "RF_importance": rf_imp.round(4)}).sort_values("mutual_info", ascending=False)
print(ranking.head(10))

scaled = pd.DataFrame(MinMaxScaler().fit_transform(X_fe_imp), columns=X_fe_imp.columns)
vt = VarianceThreshold(threshold=0.01).fit(scaled)
print("\nVarianceThreshold (0.01, skala min-max) membuang:", list(scaled.columns[~vt.get_support()]))

ranking["mutual_info"].head(10).iloc[::-1].plot(kind="barh", figsize=(7, 4), title="Mutual information dengan target (10 teratas)")
plt.tight_layout(); plt.show()

**Pengamatan:**
- **Redundansi:** fitur turunan berkorelasi tinggi dengan asalnya (`pulse_pressure` dengan `systolic_bp`, `map` dengan `diastolic_bp`, `age_x_severity` dengan `severity_score`). Contoh lain sudah muncul di 6.2: lima indikator missing per kolom berkorelasi 0,8-0,99, sehingga digabung menjadi satu (`n_missing`). Itulah *variable reduction* dengan cara paling sederhana.
- **Mutual information:** hanya `age_x_severity` (sekitar 0,035) dan `severity_score` (sekitar 0,014) yang menonjol. Sisanya di bawah 0,01, setara noise (estimasi MI pada nilai sekecil ini memang berisik, sehingga urutan di bawah dua teratas tidak perlu dibaca terlalu jauh).
- **VarianceThreshold sensitif skala dan outlier:** hasilnya bisa menyesatkan bila dijalankan sebelum outlier ditangani. Filter variansi sebaiknya dipakai setelah outlier ditangani.

In [ ]:
def sel_pipe(k):
    steps = [("combine", FunctionTransformer(add_combination)), ("flags", FunctionTransformer(add_missing_flags)),
             ("prep", make_preprocessor(imputer=KNNImputer(n_neighbors=5)))]
    if k is not None:
        steps.append(("select", SelectKBest(lambda a, b: mutual_info_classif(a, b, random_state=RANDOM_STATE), k=k)))
    steps.append(("model", lr_model()))
    return Pipeline(steps)

rows = []
for k in [4, 6, 8, 12, None]:
    m, s = cv_acc(lambda k=k: sel_pipe(k), X_tr_f, y_tr)
    label = f"Top-{k} fitur (mutual information)" if k else "Semua fitur"
    rows.append({"Seleksi fitur": label, "CV accuracy": round(m, 4), "Std": round(s, 4)})
    log_result("8.1 Seleksi fitur", label, m, s)
pd.DataFrame(rows).set_index("Seleksi fitur")

Memilih sedikit fitur terbaik memberi accuracy setara dengan semua fitur, artinya model **hanya membutuhkan segelintir fitur**. Seleksi dilakukan di dalam pipeline sehingga bebas leakage.

### 8.2 Variable Reduction (PCA)

PCA menggabungkan fitur berkorelasi menjadi beberapa komponen yang menyimpan sebagian besar variansi. Berbeda dari seleksi, PCA **membentuk fitur baru** (kurang interpretabel).

In [ ]:
prep_pca = Pipeline([("combine", FunctionTransformer(add_combination)), ("flags", FunctionTransformer(add_missing_flags)),
                     ("prep", make_preprocessor(imputer=KNNImputer(n_neighbors=5)))])
Z = prep_pca.fit_transform(X_tr_f)
cum = np.cumsum(PCA().fit(Z).explained_variance_ratio_)
print("Komponen untuk 90% variansi:", int(np.argmax(cum >= 0.90)) + 1,
      "| 95%:", int(np.argmax(cum >= 0.95)) + 1, "| total fitur:", Z.shape[1])

plt.figure(figsize=(6, 3.5))
plt.plot(range(1, len(cum) + 1), cum, marker="o")
plt.axhline(0.95, color="gray", linestyle="--")
plt.xlabel("Jumlah komponen"); plt.ylabel("Variansi kumulatif"); plt.title("PCA")
plt.tight_layout(); plt.show()

In [ ]:
def pca_pipe(n):
    steps = [("combine", FunctionTransformer(add_combination)), ("flags", FunctionTransformer(add_missing_flags)),
             ("prep", make_preprocessor(imputer=KNNImputer(n_neighbors=5)))]
    if n is not None:
        steps.append(("pca", PCA(n_components=n, random_state=RANDOM_STATE)))
    steps.append(("model", lr_model()))
    return Pipeline(steps)

rows = []
for n, label in [(5, "PCA 5 komponen"), (8, "PCA 8 komponen"), (0.95, "PCA 95% variansi"), (None, "Tanpa PCA")]:
    m, s = cv_acc(lambda n=n: pca_pipe(n), X_tr_f, y_tr)
    rows.append({"Reduksi": label, "CV accuracy": round(m, 4), "Std": round(s, 4)})
    log_result("8.2 PCA", label, m, s)
pd.DataFrame(rows).set_index("Reduksi")

Perlu sekitar 14 dari 17 komponen hanya untuk menjelaskan 90% variansi, artinya fitur-fiturnya sebagian besar **tidak saling berkorelasi**, jadi PCA hampir tidak bisa meringkasnya.

**Keputusan:** seleksi fitur dan PCA **tidak dimasukkan** ke pipeline akhir: keduanya tidak menaikkan accuracy secara jelas, sementara PCA mengorbankan interpretabilitas (padahal kita ingin menjelaskan koefisien di bagian 9).

> **Poin presentasi:** hasil reduksi menegaskan satu hal: informasi prediktif terkonsentrasi di segelintir kolom.

## 9. Pipeline Akhir, Evaluasi, dan Interpretasi

### Rekap keputusan

| Tahap | Keputusan |
|---|---|
| Duplikat | Buang 77 baris kembar (58 persis + 19 hampir sama) |
| Noise ekstrem | Buang 92 baris rusak satu blok |
| Tekanan darah tidak konsisten | Jadikan NaN lalu imputasi |
| Missing value | KNNImputer (k=5) + satu indikator `n_missing` |
| Outlier | Log1p (creatinine, wbc_count) + winsorizing 1%-99% |
| Scaling | StandardScaler (z-score) |
| Fitur baru | Interaksi `age_x_severity`, `vent_x_severity`; indikator `n_missing` |
| Seleksi / PCA | Tidak dipakai |
| Model | Logistic Regression |

In [ ]:
def final_pipe():
    return Pipeline([
        ("fix",     FunctionTransformer(fix_values)),            # BP tidak konsisten -> NaN (+ jaring pengaman rentang)
        ("combine", FunctionTransformer(add_combination)),       # fitur interaksi
        ("flags",   FunctionTransformer(add_missing_flags)),     # indikator missing
        ("prep",    make_preprocessor(imputer=KNNImputer(n_neighbors=5), outlier="winsor", log=True)),
        ("model",   lr_model()),
    ])

def evaluate(pipe, Xtr, ytr, Xte, yte):
    pipe.fit(Xtr, ytr)
    pred, proba = pipe.predict(Xte), pipe.predict_proba(Xte)[:, 1]
    return {"Accuracy": accuracy_score(yte, pred), "Precision": precision_score(yte, pred),
            "Recall": recall_score(yte, pred), "F1": f1_score(yte, pred), "ROC-AUC": roc_auc_score(yte, proba)}, pred

### 9.1 Sebelum vs Sesudah Preprocessing

Keduanya diuji pada **pasien yang sama** (test set bersih, 20%):
- **Sebelum:** model dilatih pada data mentah apa adanya (termasuk baris rusak, duplikat, dan nilai tidak konsisten), hanya diimputasi median.
- **Sesudah:** model dilatih pada data yang sudah di-preprocessing penuh.

In [ ]:
# Data latih "mentah": semua baris asli yang BUKAN bagian test (termasuk baris rusak dan duplikat)
raw_train = df.loc[~df.index.isin(X_te.index)]
X_raw_tr, y_raw_tr = raw_train[FEATURES], raw_train["survived_icu"]
print("Data latih mentah:", X_raw_tr.shape, "| data latih bersih:", X_tr.shape, "| test (sama untuk keduanya):", X_te.shape)

before_pipe = Pipeline([("imp", SimpleImputer(strategy="median")), ("model", lr_model())])
before, pred_before = evaluate(before_pipe, X_raw_tr, y_raw_tr, X_te, y_te)
after, pred_after = evaluate(final_pipe(), X_tr, y_tr, X_te, y_te)

result = pd.DataFrame({"Sebelum (data mentah)": before, "Sesudah (preprocessing penuh)": after}).T.round(4)
n_te = len(y_te); p = y_te.value_counts(normalize=True).max()
print("Accuracy menebak kelas mayoritas:", round(p, 4))
print("Galat baku accuracy pada", n_te, "pasien: sekitar", round(np.sqrt(0.67 * 0.33 / n_te), 4), "(selisih lebih kecil dari ini tidak bermakna)")
result

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
vals = [before["Accuracy"], after["Accuracy"]]
ax[0].bar(["Sebelum", "Sesudah"], vals, color=["tab:gray", "tab:blue"])
ax[0].axhline(p, color="red", linestyle="--", label=f"Tebak kelas mayoritas ({p:.3f})")
ax[0].set_ylim(0.5, 0.75); ax[0].set_ylabel("Accuracy"); ax[0].set_title("Accuracy pada test set"); ax[0].legend()
for i, v in enumerate(vals): ax[0].text(i, v + 0.003, f"{v:.3f}", ha="center")

ConfusionMatrixDisplay(confusion_matrix(y_te, pred_after), display_labels=["Tidak selamat", "Selamat"]).plot(ax=ax[1], cmap="Blues", colorbar=False)
ax[1].set_title("Confusion matrix (sesudah preprocessing)")
plt.tight_layout(); plt.show()

### 9.2 Validasi Lebih Stabil: 5-fold Cross-Validation

Satu kali split bisa kebetulan. CV pada data latih bersih (preprocessing dihitung ulang di tiap fold) memberi gambaran yang lebih stabil.

In [ ]:
m, s = cv_acc(final_pipe, X_tr, y_tr)
log_result("9. Final", "Pipeline akhir (CV pada data latih)", m, s)
print(f"Pipeline akhir, 5-fold CV: accuracy = {m:.4f} +/- {s:.4f}")
print(f"Pembanding: tebak kelas mayoritas pada data latih = {y_tr.value_counts(normalize=True).max():.4f}")

### 9.3 Interpretasi: Apa yang Dipelajari Model?

Karena modelnya Logistic Regression, koefisien bisa dibaca langsung. Semua fitur sudah distandarkan, sehingga besarnya koefisien **dapat dibandingkan antar fitur**. Tanda negatif berarti semakin besar nilai fitur, peluang selamat semakin kecil.

In [ ]:
pipe = final_pipe().fit(X_tr, y_tr)

front = Pipeline(pipe.steps[:3])                                  # fix -> combine -> flags
cols = list(front.transform(X_tr.head()).columns)
names = SKEWED + [c for c in cols if c not in SKEWED]             # urutan kolom setelah langkah 'log'
coef = pd.Series(pipe.named_steps["model"].coef_[0], index=names).sort_values(key=np.abs, ascending=False)

top = coef.head(10)
top.iloc[::-1].plot(kind="barh", figsize=(7, 4.5), color=["tab:red" if v < 0 else "tab:green" for v in top.iloc[::-1]])
plt.title("10 koefisien terbesar (fitur distandarkan)")
plt.axvline(0, color="black", linewidth=0.8); plt.tight_layout(); plt.show()
coef.head(10).round(3).to_frame("koefisien")

**Cara membaca:** fitur teratas hampir pasti `age`, `severity_score`, dan interaksinya. Ini sama dengan temuan korelasi (4.4), tabel agregasi (7.4), dan mutual information (8.1): ketiganya menunjuk ke kolom yang sama. Kolom tanda vital dan lab punya koefisien mendekati nol.

### 9.4 Ringkasan Seluruh Eksperimen

In [ ]:
pd.set_option("display.max_rows", 200)
pd.DataFrame(EXPERIMENT_LOG)

## 10. Kesimpulan

*(Cek angka dengan output yang kamu dapat saat menjalankan notebook; hasil bisa sedikit berbeda antar versi library.)*

**Temuan kualitas data.** Dataset memuat enam jenis kekotoran: duplikat tersembunyi (77 baris, termasuk 19 yang hanya terlihat setelah NaN diabaikan), 92 baris rusak satu blok, tekanan darah tidak konsisten, missing value berkelompok (MCAR), outlier/distribusi miring, dan capping pada saturasi oksigen.

**Cakupan preprocessing.** Seluruh materi slide diterapkan: cleaning (noise, duplicate, missing, outlier), transformation (encoding, scaling, smoothing, aggregation, discretization, feature engineering), dan reduction (seleksi fitur, PCA). Setiap teknik yang berpengaruh pada model diuji dengan cross-validation dan dicatat di tabel ringkasan.

**Hasil.** Accuracy semua perlakuan berada di kisaran 0,66-0,68, hanya sedikit di atas menebak kelas mayoritas (64%). Selisih antar teknik lebih kecil daripada simpangan baku antar-fold dan galat baku test set (sekitar 1,6 poin), sehingga tidak ada teknik yang terbukti lebih baik.

**Penjelasan.** Hanya empat kolom yang berhubungan dengan target: `age`, `severity_score`, `ventilation_required`, dan `chronic_conditions`. Kolom yang paling kotor (tanda vital dan lab) hampir tidak berkorelasi dengan target. Membersihkannya membuat data **valid dan dapat dipercaya**, tetapi tidak menambah informasi yang memang tidak ada. Eksperimen di bagian 5.1 memperkuat ini: menambahkan kembali baris rusak dan duplikat ke data latih tidak mengubah accuracy secara terukur.

**Pelajaran.** *Garbage in, garbage out* berlaku dua arah: preprocessing tidak bisa menciptakan sinyal yang tidak ada. Nilai preprocessing ada pada validitas data, evaluasi yang bebas leakage, dan pipeline yang bisa dipakai ulang.

**Keterbatasan yang perlu disebut.**
- Perlakuan tekanan darah tidak konsisten (jadikan NaN) adalah asumsi; nilai aslinya tidak diketahui.
- MCAR tidak bisa dibuktikan, hanya didukung oleh uji statistik.
- Dataset tampak sintetis (pola kerusakan terlalu rapi), sehingga hasilnya belum tentu berlaku pada data rumah sakit sungguhan.